# RF-DETR two-stage people counting (ER FlowScan)

A clean, run-top-to-bottom pipeline for **counting and tracking people** in crowded / top-down footage, with optional pose on the foreground.

**Pipeline**
1. **Detect** every person (RF-DETR large), with **tiled inference** so small/distant people are found.
2. **Track** them through a motion + appearance (**ReID**) tracker → stable ids and a stable count.
3. **Pose** the N largest (foreground) people at high precision.

Outputs an annotated MP4 with id-colored boxes, a live `count`, skeletons on the foreground, and prints the average people/frame and the total unique ids.

> `Runtime → Change runtime type → GPU` first. The full pipeline is an **offline** tool (tiling + pose ≈ 1-2 FPS); see the speed presets at the end.

In [ ]:
!nvidia-smi -L || echo 'No GPU - Runtime > Change runtime type > GPU.'

## 1. Install

In [ ]:
import os

REPO = '/content/rf-detr'
if not os.path.exists(REPO):
    !git clone --branch develop https://github.com/shingo257/rf-detr.git {REPO}
%cd {REPO}
!git pull --ff-only
# .[reid] pulls onnxruntime (needed for the ReID embedding backend); onnxscript is for the export in step 3.
!pip -q install -e '.[reid]' onnx onnxscript
print('\nReady in', REPO)

## 2. Choose a video

Defaults to a public people-walking clip. Uncomment the upload lines for your own footage.

In [ ]:
import os, cv2

VIDEO = 'demo.mp4'
if not os.path.exists(VIDEO):
    !wget -q -O {VIDEO} https://media.roboflow.com/supervision/video-examples/people-walking.mp4

# from google.colab import files
# up = files.upload(); VIDEO = next(iter(up))

cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f'Could not open {VIDEO}'
print(f'{VIDEO}: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames, {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')
cap.release()

## 3. ReID embedding model

A learned appearance embedding keeps ids stable across occlusions and works on **grayscale/IR** footage (unlike a color histogram). This exports a small MobileNetV3 to `reid.onnx` (torchvision weights, always available). For best quality swap in a person-ReID OSNet ONNX with the same 1x3x256x128 input.

In [ ]:
import torch, torch.nn as nn, torchvision

backbone = torchvision.models.mobilenet_v3_small(weights='DEFAULT')
extractor = nn.Sequential(backbone.features, nn.AdaptiveAvgPool2d(1), nn.Flatten()).eval()
torch.onnx.export(extractor, torch.randn(1, 3, 256, 128), 'reid.onnx',
                  input_names=['input'], output_names=['output'], opset_version=18)
print('wrote reid.onnx')

## 4. Run the full pipeline

Detect (tiled) + track + ReID + pose on the foreground 8. Watch the printed `instances / frames` (average people) and `Unique track ids over clip`. Lower `--max-frames` while experimenting.

In [ ]:
!rfdetr-demo video --task detect --person-only --track --model large \
    --resolution 960 --threshold 0.25 \
    --tile 640 --tile-overlap 256 \
    --reid-model reid.onnx --reid-similarity 0.6 \
    --pose-topk 8 \
    --source {VIDEO} --max-frames 120 --output flowscan.mp4

import os
from base64 import b64encode
from IPython.display import HTML, display
os.system('ffmpeg -y -loglevel error -i flowscan.mp4 -vcodec libx264 -pix_fmt yuv420p flowscan_h264.mp4')
display(HTML(f'<video width=820 controls><source src="data:video/mp4;base64,'
             f'{b64encode(open("flowscan_h264.mp4", "rb").read()).decode()}" type="video/mp4"></video>'))

## 5. Speed presets

The full pipeline is offline (~1-2 FPS). Trim stages for speed:

| Goal | Command changes | approx FPS |
| --- | --- | --- |
| **Full (most accurate)** | as above | 1-2 |
| **Count + track only** (fast) | drop `--tile`, `--pose-topk`; keep `--reid-model` | ~5 |
| **Fastest count** | drop `--tile`, `--pose-topk`, `--reid-model` | ~10 |
| **Recover ReID speed** | add `--reid-stride 3` | +~1.5x |
| **Fewer boundary doubles** | raise `--tile-overlap` (256-> 320) | slower |

### Knobs
| Flag | Meaning | Default |
| --- | --- | --- |
| `--resolution` | detector input size (recall on small people) | 576 |
| `--threshold` | detection confidence (lower = more recall) | 0.25 here |
| `--tile` / `--tile-overlap` | tiled inference for distant people | off / 128 |
| `--reid-model` / `--reid-similarity` | appearance ReID (id stability) | off / 0.6 |
| `--reid-stride` | run the embedding every Nth frame | 1 |
| `--pose-topk` | pose the N largest tracked people | 0 |

For deeper analysis (ReID sweeps, per-stage comparisons) see `reid_comparison_colab.ipynb`.